# Debug: Entrenamiento y Exportación YOLOv8
## Álvaro Zarabanda - 20251595006

Este notebook permite ejecutar paso a paso el entrenamiento y exportación del modelo YOLO para identificar exactamente dónde ocurren los errores.

## 1. Importar Librerías y Configuración

In [9]:
from ultralytics import YOLO
import torch
from pathlib import Path
import os
import sys
import traceback

# Configuración
DATA_YAML = "dataset/data.yaml"
MODEL_NAME = "yolov8n.pt"
EPOCHS = 25
BATCH_SIZE = 16
IMG_SIZE = 640

print(" Librerías importadas correctamente")
print(f" Python: {sys.version}")
print(f" PyTorch: {torch.__version__}")
print(f" Ultralytics: {YOLO.__module__}")
print(f"  Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

 Librerías importadas correctamente
 Python: 3.13.9 (main, Oct 14 2025, 00:00:00) [GCC 15.2.1 20250808 (Red Hat 15.2.1-1)]
 PyTorch: 2.9.1+cu128
 Ultralytics: ultralytics.models.yolo.model
  Device: CPU


## 2. Verificar Dataset

In [10]:
# Verificar existencia de archivos
dataset_path = Path(DATA_YAML)

if dataset_path.exists():
    print(f" Dataset YAML encontrado: {dataset_path}")
    
    # Leer contenido
    with open(dataset_path, 'r') as f:
        content = f.read()
    print("\n Contenido del data.yaml:")
    print(content)
    
    # Verificar imágenes
    train_dir = Path("dataset/images/train")
    val_dir = Path("dataset/images/val")
    
    train_imgs = len(list(train_dir.glob("*.jpg"))) if train_dir.exists() else 0
    val_imgs = len(list(val_dir.glob("*.jpg"))) if val_dir.exists() else 0
    
    print(f"\n Estadísticas:")
    print(f"   - Imágenes train: {train_imgs}")
    print(f"   - Imágenes val: {val_imgs}")
else:
    print(f" Dataset YAML NO encontrado: {dataset_path}")

 Dataset YAML encontrado: dataset/data.yaml

 Contenido del data.yaml:
# Dataset de objetos del salón
# Auto-generado con bounding boxes automáticos

path: /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset
train: images/train
val: images/val

# Clases
nc: 6
names: ['cpu', 'mesa', 'mouse', 'pantalla', 'silla', 'teclado']


 Estadísticas:
   - Imágenes train: 1238
   - Imágenes val: 312


## 3. Cargar Modelo YOLOv8n

In [11]:
try:
    print(f" Cargando modelo: {MODEL_NAME}")
    model = YOLO(MODEL_NAME)
    print(" Modelo cargado exitosamente")
    print(f" Resumen del modelo:")
    print(f"   - Nombre: {model.model_name}")
    print(f"   - Tarea: {model.task}")
except Exception as e:
    print(f" Error al cargar modelo: {e}")
    traceback.print_exc()

 Cargando modelo: yolov8n.pt
 Modelo cargado exitosamente
 Resumen del modelo:
   - Nombre: yolov8n.pt
   - Tarea: detect


## 4. Entrenar Modelo

In [12]:
try:
    print(" Iniciando entrenamiento...")
    print(f"   - Épocas: {EPOCHS}")
    print(f"   - Batch size: {BATCH_SIZE}")
    print(f"   - Tamaño imagen: {IMG_SIZE}")
    
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device='cpu', 
        project="runs/detect",
        name="inventario",
        exist_ok=True,
        
        # Configuración básica
        patience=20,
        save=True,
        verbose=True,
        plots=True
    )
    
    print("\n Entrenamiento completado!")
    print(f" Resultados guardados en: {results.save_dir}")
    
except Exception as e:
    print(f"\n Error durante el entrenamiento: {e}")
    traceback.print_exc()

 Iniciando entrenamiento...
   - Épocas: 25
   - Batch size: 16
   - Tamaño imagen: 640
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=inventario, nbs=64, nms=False, opset=None, optimize=False, optimiz

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/runs/detect/inventario
Starting training for 25 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/25         0G      0.741      2.807      1.395         15        640: 100% ━━━━━━━━━━━━ 78/78 1.4s/it 1:490.9sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2s/it 12.0s.3s
                   all        312        312      0.436      0.536      0.478      0.315

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/25    

## 5. Validar Modelo Entrenado

In [13]:
try:
    print(" Validando modelo...")
    metrics = model.val()
    
    print("\n MÉTRICAS FINALES:")
    print(f"   - mAP50: {metrics.box.map50:.4f}")
    print(f"   - mAP50-95: {metrics.box.map:.4f}")
    print(f"   - Precisión: {metrics.box.mp:.4f}")
    print(f"   - Recall: {metrics.box.mr:.4f}")
    
    print("\n Validación completada")
    
except Exception as e:
    print(f"\n Error durante la validación: {e}")
    traceback.print_exc()

 Validando modelo...
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5759.6±968.5 MB/s, size: 111.8 KB)
val: Scanning /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/dataset/labels/val.cache... 312 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 312/312 1.5Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 2.4it/s 8.4s0.4s
                   all        312        312      0.928      0.817      0.913      0.799
                   cpu         42         42       0.94      0.746      0.865       0.79
                  mesa         56         56       0.92      0.911      0.981       0.88
                 mouse         53         53      0.929      0.742      0.834      0.697
              pantalla         51         51      0.9

## 6. Verificar Modelo Guardado

In [14]:
try:
    # Buscar el modelo guardado
    model_path = Path("runs/detect/inventario/weights/best.pt")
    
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024 * 1024)
        print(f" Modelo encontrado: {model_path}")
        print(f" Tamaño: {size_mb:.2f} MB")
        
        # Listar todos los archivos en weights
        weights_dir = model_path.parent
        print(f"\n Archivos en {weights_dir}:")
        for f in weights_dir.iterdir():
            if f.is_file():
                size = f.stat().st_size / (1024 * 1024)
                print(f"   - {f.name}: {size:.2f} MB")
    else:
        print(f" Modelo NO encontrado en: {model_path}")
        
except Exception as e:
    print(f" Error verificando modelo: {e}")
    traceback.print_exc()

 Modelo encontrado: runs/detect/inventario/weights/best.pt
 Tamaño: 5.96 MB

 Archivos en runs/detect/inventario/weights:
   - last.pt: 5.96 MB
   - best.pt: 5.96 MB
   - best.onnx: 11.70 MB


## 7. Exportar a ONNX

In [15]:
try:
    print(" Exportando modelo a ONNX...")
    
    # Cargar el mejor modelo
    best_model = YOLO("runs/detect/inventario/weights/best.pt")
    
    # Exportar a ONNX
    onnx_path = best_model.export(
        format='onnx',
        imgsz=640,
        simplify=True,
        opset=12  # Versión de ONNX compatible
    )
    
    print(f"\nModelo ONNX exportado: {onnx_path}")
    
    # Verificar tamaño
    if Path(onnx_path).exists():
        size_mb = Path(onnx_path).stat().st_size / (1024 * 1024)
        print(f" Tamaño ONNX: {size_mb:.2f} MB")
    
except Exception as e:
    print(f"\n Error durante la exportación a ONNX:")
    print(f"   Tipo de error: {type(e).__name__}")
    print(f"   Mensaje: {str(e)}")
    print("\n Stack trace completo:")
    traceback.print_exc()

 Exportando modelo a ONNX...
Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.9.1+cu128 CPU (Intel Core i7-14700)
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs



PyTorch: starting from 'runs/detect/inventario/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)

ONNX: starting export with onnx 1.19.1 opset 12...
ONNX: slimming with onnxslim 0.1.74...
ONNX: export success ✅ 0.5s, saved as 'runs/detect/inventario/weights/best.onnx' (11.7 MB)

Export complete (0.6s)
Results saved to /run/media/SoporteOATI/HDD/Maestria/Repositorios/BigData-CNN/inventario/runs/detect/inventario/weights
Predict:         yolo predict task=detect model=runs/detect/inventario/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs/detect/inventario/weights/best.onnx imgsz=640 data=dataset/data.yaml  
Visualize:       https://netron.app

Modelo ONNX exportado: runs/detect/inventario/weights/best.onnx
 Tamaño ONNX: 11.70 MB


## 8. Resumen y Verificación Final

In [16]:
import glob

print("="*60)
print("  RESUMEN FINAL")
print("="*60)

# Buscar todos los modelos generados
pt_models = glob.glob("runs/detect/*/weights/*.pt")
onnx_models = glob.glob("runs/detect/*/weights/*.onnx")

print("\n Modelos PyTorch (.pt):")
if pt_models:
    for model in pt_models:
        size_mb = Path(model).stat().st_size / (1024 * 1024)
        print(f"    {model} ({size_mb:.2f} MB)")
else:
    print("    No se encontraron modelos .pt")

print("\n Modelos ONNX (.onnx):")
if onnx_models:
    for model in onnx_models:
        size_mb = Path(model).stat().st_size / (1024 * 1024)
        print(f"    {model} ({size_mb:.2f} MB)")
else:
    print("    No se encontraron modelos .onnx")



  RESUMEN FINAL

 Modelos PyTorch (.pt):
    runs/detect/inventario/weights/last.pt (5.96 MB)
    runs/detect/inventario/weights/best.pt (5.96 MB)

 Modelos ONNX (.onnx):
    runs/detect/inventario/weights/best.onnx (11.70 MB)
